# Snake-RepairLLaMA — Baseline Evaluation

**Goal:** measure how vanilla `codellama/CodeLlama-7b-Python-hf` (no LoRA, no fine-tuning) performs on the QuixBugs (40 bugs) and BugsInPy (196 bugs) IR4/OR2 evaluation sets — using the **same prompts, sampling protocol, and metrics** that we'll use later for the trained adapter, so results are directly comparable.

**Runtime:** Colab Free T4 (16 GB VRAM). Loads CodeLlama-7B in **8-bit** (~7 GB) — fits comfortably.

**Inference protocol (matches the RepairLLaMA paper):**
- 10 candidate patches per bug
- Sampling: `do_sample=True, temperature=1.0, top_p=0.95`  (NOT beam search)
- `max_new_tokens=256`
- 8-bit quantization

**Cell-by-cell flow:**
1. Setup: clone repo, install deps
2. HF login (paste token or load from Colab secret)
3. Sanity check: load model + generate on 1 bug to verify everything works
4. Run full inference on QuixBugs (~40 bugs × 10 = 400 generations)
5. Run full inference on BugsInPy (~196 bugs × 10 = 1960 generations)
6. Score & print Top-1 / Top-3 / Top-10 exact / AST / compile / buried-fix
7. (Optional) Save results JSONL to Google Drive

## 1. Setup — clone repo & install deps

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/AliSuleman27/snake-repairllama-baseline.git"
REPO_DIR = "/content/snake-repairllama-baseline"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("cwd:", os.getcwd())
print("contents:", os.listdir(REPO_DIR))

In [ ]:
# Install deps. Colab has torch + transformers preinstalled; we add the rest.
%pip install -q -U "transformers>=4.40" "peft>=0.10" "accelerate>=0.30" "bitsandbytes>=0.43" "datasets>=2.18" tqdm

## 2. HF login

CodeLlama is gated. Add your HF token as a Colab Secret named `HF_TOKEN`, **or** paste it in the prompt below.

In [ ]:
import os

token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    pass

if not token:
    from getpass import getpass
    token = getpass("Paste your HF token (write or read access): ")

from huggingface_hub import login
login(token=token, add_to_git_credential=False)
print("HF login OK")

## 3. Sanity check — load model & test on 1 bug

Run this BEFORE the full eval. Verifies model loads, prompt is well-formed, and a generation looks like a Python snippet (not garbage). ~3 min on T4.

In [ ]:
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# NOTE: use the BASE variant, not -Python-hf. The Python variant was not trained
# for infilling and its embedding table omits the FIM token rows (<PRE>=32007,
# <SUF>=32008, <MID>=32009), so feeding our <FILL_ME> prompts triggers an
# out-of-bounds embedding lookup → CUDA device-side assert.
MODEL_NAME = "codellama/CodeLlama-7b-hf"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb = BitsAndBytesConfig(load_in_8bit=True, llm_int8_threshold=6.0)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb, device_map="auto"
)
model.eval()
print("Model loaded. VRAM:", torch.cuda.memory_allocated() / 1e9, "GB")
print("vocab_size:", model.config.vocab_size, "(should be 32016 for FIM support)")

In [ ]:
# Pick the first QuixBugs bug, generate 1 patch, eyeball the output.
with open("data/quixbugs_eval.jsonl") as f:
    bug = json.loads(f.readline())

print("=== INPUT (IR4) ===")
print(bug["input"])
print("=== GOLD (OR2) ===")
print(bug["output"])

inputs = tokenizer(bug["input"], return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(
        **inputs, max_new_tokens=256,
        do_sample=True, temperature=1.0, top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )
raw = tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("=== RAW GENERATION ===")
print(raw)

from src.postprocess import extract_patch
print("=== EXTRACTED (strict) ===")
print(extract_patch(raw, mode="strict"))

## 4. Full inference — QuixBugs (40 bugs × 10 samples)

~30 min on T4. Resumes if interrupted (re-run the cell).

In [ ]:
# Free the sanity-check model first to make room (we re-load via the helper)
import gc
del model, tokenizer
gc.collect(); torch.cuda.empty_cache()

from src.inference import run_inference

run_inference(
    eval_jsonl="data/quixbugs_eval.jsonl",
    output_jsonl="results/quixbugs_codellama_baseline.jsonl",
    model_name=MODEL_NAME,
    n_samples=10,
    load_in_8bit=True,
)

## 5. Full inference — BugsInPy (196 bugs × 10 samples)

~3-4 hours on T4 free. If your Colab session might disconnect, run with `limit=50` first to get partial numbers, or use the resume-on-rerun behavior.

In [ ]:
from src.inference import run_inference

run_inference(
    eval_jsonl="data/bugsinpy_eval.jsonl",
    output_jsonl="results/bugsinpy_codellama_baseline.jsonl",
    model_name=MODEL_NAME,
    n_samples=10,
    load_in_8bit=True,
    # limit=50,  # uncomment to do a quick partial run first
)

## 6. Score & report

Top-1 / Top-3 / Top-10 for **exact-match**, **AST-match**, **compile**, and **buried-fix** (gold appears anywhere in the generation — useful to see if the base model 'knows' the fix even if it can't isolate it cleanly).

In [ ]:
from src.metrics import evaluate_file, print_report

qb = evaluate_file(
    inference_jsonl="results/quixbugs_codellama_baseline.jsonl",
    eval_jsonl="data/quixbugs_eval.jsonl",
)
print_report("QuixBugs — vanilla CodeLlama-7B-Python (baseline)", qb)

In [ ]:
from src.metrics import evaluate_file, print_report

bp = evaluate_file(
    inference_jsonl="results/bugsinpy_codellama_baseline.jsonl",
    eval_jsonl="data/bugsinpy_eval.jsonl",
)
print_report("BugsInPy — vanilla CodeLlama-7B-Python (baseline)", bp)

## 7. (Optional) Save results to Google Drive

Persists `results/*.jsonl` so you can compare against the trained-adapter run later.

In [ ]:
from google.colab import drive
import shutil, os

drive.mount("/content/drive")
dest = "/content/drive/MyDrive/snake-repairllama-baseline-results"
os.makedirs(dest, exist_ok=True)
for fn in os.listdir("results"):
    if fn.endswith(".jsonl"):
        shutil.copy(f"results/{fn}", f"{dest}/{fn}")
        print("Saved:", fn)